# 🔋 Cross-Modal Generative AI Energy Measurement
**For Nature Communications submission**

This notebook measures energy consumption across 4 generative AI modalities:
- 📝 Text (Phi-3-mini)
- 🖼️ Image (Stable Diffusion v1.5)
- 🎬 Video (AnimateDiff)
- 🎵 Music (MusicGen small/medium)

**GPU Power Measurement**: NVIDIA `pynvml` (real-time GPU power draw)

⚠️ **Important**: Go to Runtime → Change runtime type → Select **GPU** (T4 or A100)

In [ ]:
# ============================================================
# Cell 1: Install dependencies
# ============================================================
!pip install -q transformers==4.47.1 diffusers accelerate pynvml scipy
!pip install -q sentencepiece protobuf

In [ ]:
# ============================================================
# Cell 2: Mount Drive + Setup
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/project_energy_results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results will be saved to: {RESULTS_DIR}')

In [ ]:
# ============================================================
# Cell 3: GPU Info + Verify Setup
# ============================================================
import torch
import pynvml

pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)
gpu_name = pynvml.nvmlDeviceGetName(handle)
gpu_mem = pynvml.nvmlDeviceGetMemoryInfo(handle)

print(f'🖥️  GPU: {gpu_name}')
print(f'💾 Memory: {gpu_mem.total / 1e9:.1f} GB')
print(f'🔥 CUDA available: {torch.cuda.is_available()}')
print(f'📦 PyTorch: {torch.__version__}')

# Quick power reading test
power_mw = pynvml.nvmlDeviceGetPowerUsage(handle)
print(f'⚡ Current GPU power: {power_mw/1000:.1f} W')

DEVICE = 'cuda'
GPU_NAME = gpu_name if isinstance(gpu_name, str) else gpu_name.decode('utf-8')

In [ ]:
# ============================================================
# Cell 4: Power Monitor (NVIDIA pynvml)
# ============================================================
import time, json, threading, gc
import numpy as np

REPEATS = 30

class NVIDIAPowerMonitor:
    """GPU power monitor using NVIDIA pynvml.
    Samples GPU power draw at ~100ms intervals."""
    
    def __init__(self, gpu_index=0, interval_ms=100):
        pynvml.nvmlInit()
        self.handle = pynvml.nvmlDeviceGetHandleByIndex(gpu_index)
        self.interval = interval_ms / 1000.0
        self.readings = []
        self._running = False
    
    def start(self):
        self.readings = []
        self._running = True
        self._start = time.time()
        
        def _sample():
            while self._running:
                try:
                    power_mw = pynvml.nvmlDeviceGetPowerUsage(self.handle)
                    self.readings.append((time.time() - self._start, power_mw / 1000.0))
                except pynvml.NVMLError:
                    pass
                time.sleep(self.interval)
        
        self._thread = threading.Thread(target=_sample, daemon=True)
        self._thread.start()
    
    def stop(self):
        self._running = False
        self._thread.join(timeout=3)
        
        if not self.readings or len(self.readings) < 2:
            return {'total_energy_j': 0, 'avg_power_w': 0, 'max_power_w': 0,
                    'duration_sec': 0, 'samples': 0}
        
        powers = [r[1] for r in self.readings]
        duration = self.readings[-1][0] - self.readings[0][0]
        # Trapezoidal integration for energy (Joules)
        energy_j = sum(
            (self.readings[i][1] + self.readings[i-1][1]) / 2 *
            (self.readings[i][0] - self.readings[i-1][0])
            for i in range(1, len(self.readings))
        )
        return {
            'total_energy_j': round(energy_j, 2),
            'avg_power_w': round(np.mean(powers), 2),
            'max_power_w': round(max(powers), 2),
            'duration_sec': round(duration, 2),
            'samples': len(self.readings)
        }


def run_and_measure(task_fn, pm, cooldown=3):
    """Run task with power measurement."""
    torch.cuda.synchronize()
    time.sleep(cooldown)
    pm.start()
    time.sleep(0.5)  # stabilize
    t0 = time.time()
    extra = task_fn()
    torch.cuda.synchronize()
    gen_time = time.time() - t0
    time.sleep(0.5)
    result = pm.stop()
    result['generation_time_sec'] = round(gen_time, 2)
    if extra:
        result.update(extra)
    return result


def save_results(data, filename):
    path = os.path.join(RESULTS_DIR, filename)
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)
    # Also save locally
    with open(f'/content/{filename}', 'w') as f:
        json.dump(data, f, indent=2)
    print(f'  💾 Saved {filename} ({len(data)} records)')


def print_summary(results, key='total_energy_j'):
    vals = [r[key] for r in results if r.get(key, 0) > 0]
    if vals:
        print(f'  📊 {key}: {np.mean(vals):.1f} ± {np.std(vals):.1f} (n={len(vals)})')


print('✅ Power monitor ready!')
# Quick test
pm_test = NVIDIAPowerMonitor()
pm_test.start()
time.sleep(2)
r = pm_test.stop()
print(f'   Test: {r["samples"]} samples, {r["avg_power_w"]:.1f}W avg, {r["total_energy_j"]:.1f}J')

In [ ]:
# ============================================================
# Cell 5: EXPERIMENT 1 — TEXT GENERATION
# ============================================================
from transformers import AutoModelForCausalLM, AutoTokenizer

print('\n' + '='*60)
print('📝 EXPERIMENT 1: TEXT GENERATION')
print('='*60)

tokenizer = AutoTokenizer.from_pretrained('microsoft/Phi-3-mini-4k-instruct')
model = AutoModelForCausalLM.from_pretrained(
    'microsoft/Phi-3-mini-4k-instruct', torch_dtype=torch.float16, device_map=DEVICE)
params = sum(p.numel() for p in model.parameters())
print(f'  Model: Phi-3-mini ({params/1e9:.1f}B) on {DEVICE} ({GPU_NAME})')

# Warm up
inp = tokenizer('Hello', return_tensors='pt').to(DEVICE)
with torch.no_grad(): model.generate(**inp, max_new_tokens=16)
torch.cuda.synchronize()

prompt = 'Explain the impact of artificial intelligence on renewable energy systems in detail.'
token_configs = [64, 128, 256, 512]
text_results = []
pm = NVIDIAPowerMonitor()

for max_tok in token_configs:
    print(f'\n  📏 {max_tok} tokens × {REPEATS} runs')
    for run in range(REPEATS):
        def task(mt=max_tok):
            inp = tokenizer(prompt, return_tensors='pt').to(DEVICE)
            with torch.no_grad():
                out = model.generate(**inp, max_new_tokens=mt, do_sample=True, temperature=0.7)
            n_tok = out.shape[1] - inp['input_ids'].shape[1]
            return {'tokens_generated': int(n_tok)}
        
        result = run_and_measure(task, pm, cooldown=2)
        result['modality'] = 'text'
        result['model'] = 'Phi-3-mini-4k'
        result['params_B'] = round(params/1e9, 1)
        result['max_tokens'] = max_tok
        result['run'] = run + 1
        result['hardware'] = GPU_NAME
        result['backend'] = 'cuda'
        text_results.append(result)
        
        if (run+1) % 10 == 0 or run == 0:
            print(f'    Run {run+1}/{REPEATS}: {result["total_energy_j"]:.0f}J | {result["avg_power_w"]:.1f}W | {result["generation_time_sec"]:.1f}s')
    
    print_summary([r for r in text_results if r['max_tokens']==max_tok])

save_results(text_results, f'exp1_text_{GPU_NAME.replace(" ", "_")}.json')
del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
print('\n✅ Text experiment done!')

In [ ]:
# ============================================================
# Cell 6: EXPERIMENT 2 — IMAGE GENERATION
# ============================================================
from diffusers import StableDiffusionPipeline

print('\n' + '='*60)
print('🖼️  EXPERIMENT 2: IMAGE GENERATION')
print('='*60)

pipe = StableDiffusionPipeline.from_pretrained(
    'sd-legacy/stable-diffusion-v1-5', torch_dtype=torch.float16, safety_checker=None)
pipe = pipe.to(DEVICE)
params = sum(p.numel() for p in pipe.unet.parameters())
print(f'  Model: SD v1.5 (UNet {params/1e9:.1f}B) on {DEVICE} ({GPU_NAME})')

# Warm up
with torch.no_grad(): pipe('test', num_inference_steps=3, height=256, width=256)
torch.cuda.synchronize()

prompt = 'A beautiful sunset over mountains with a river, photorealistic'
configs = [
    (256, 20), (256, 30),
    (384, 20), (384, 30),
    (512, 20), (512, 30),
]
image_results = []
pm = NVIDIAPowerMonitor()

for res, steps in configs:
    print(f'\n  📏 {res}x{res}, {steps} steps × {REPEATS} runs')
    for run in range(REPEATS):
        def task(r=res, s=steps):
            with torch.no_grad():
                pipe(prompt, num_inference_steps=s, height=r, width=r)
            return {'resolution': f'{r}x{r}', 'steps': s}
        
        result = run_and_measure(task, pm, cooldown=2)
        result['modality'] = 'image'
        result['model'] = 'SD-v1-5'
        result['params_B'] = round(params/1e9, 1)
        result['resolution'] = f'{res}x{res}'
        result['steps'] = steps
        result['run'] = run + 1
        result['hardware'] = GPU_NAME
        result['backend'] = 'cuda'
        image_results.append(result)
        
        if (run+1) % 10 == 0 or run == 0:
            print(f'    Run {run+1}/{REPEATS}: {result["total_energy_j"]:.0f}J | {result["avg_power_w"]:.1f}W | {result["generation_time_sec"]:.1f}s')
    
    print_summary([r for r in image_results if r['resolution']==f'{res}x{res}' and r['steps']==steps])

save_results(image_results, f'exp2_image_{GPU_NAME.replace(" ", "_")}.json')
del pipe; gc.collect(); torch.cuda.empty_cache()
print('\n✅ Image experiment done!')

In [ ]:
# ============================================================
# Cell 7: EXPERIMENT 3 — VIDEO GENERATION
# ============================================================
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler

print('\n' + '='*60)
print('🎬 EXPERIMENT 3: VIDEO GENERATION')
print('='*60)

adapter = MotionAdapter.from_pretrained(
    'guoyww/animatediff-motion-adapter-v1-5-2', torch_dtype=torch.float16)
pipe = AnimateDiffPipeline.from_pretrained(
    'sd-legacy/stable-diffusion-v1-5', motion_adapter=adapter, torch_dtype=torch.float16)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config, beta_schedule='linear')
pipe = pipe.to(DEVICE)
print(f'  Model: AnimateDiff on {DEVICE} ({GPU_NAME})')

# Warm up
with torch.no_grad(): pipe('test', num_frames=4, num_inference_steps=3, height=256, width=256)
torch.cuda.synchronize()

prompt = 'A beautiful sunset over mountains, timelapse, cinematic'
# CUDA has more VRAM — can try 512 too
configs = [
    (4, 20, 256), (8, 20, 256), (12, 20, 256),
    (4, 30, 256), (8, 30, 256),
]

# Check if we have enough VRAM for 512x512
gpu_mem = torch.cuda.get_device_properties(0).total_memory
if gpu_mem > 20e9:  # >20GB (A100)
    configs.extend([(4, 20, 512), (8, 20, 512)])
    print('  💪 A100 detected — adding 512x512 configs')

video_results = []
pm = NVIDIAPowerMonitor()

for frames, steps, res in configs:
    print(f'\n  📏 {frames} frames, {res}x{res}, {steps} steps × {REPEATS} runs')
    for run in range(REPEATS):
        def task(f=frames, s=steps, r=res):
            with torch.no_grad():
                pipe(prompt, num_frames=f, num_inference_steps=s, height=r, width=r)
            return {'frames': f, 'steps': s, 'resolution': f'{r}x{r}'}
        
        try:
            result = run_and_measure(task, pm, cooldown=3)
        except RuntimeError as e:
            if 'out of memory' in str(e):
                print(f'    ⚠️ OOM at {frames}f/{res}px — skipping')
                torch.cuda.empty_cache()
                break
            raise
        
        result['modality'] = 'video'
        result['model'] = 'AnimateDiff-v1.5'
        result['resolution'] = f'{res}x{res}'
        result['frames'] = frames
        result['steps'] = steps
        result['run'] = run + 1
        result['hardware'] = GPU_NAME
        result['backend'] = 'cuda'
        video_results.append(result)
        
        if (run+1) % 10 == 0 or run == 0:
            print(f'    Run {run+1}/{REPEATS}: {result["total_energy_j"]:.0f}J | {result["avg_power_w"]:.1f}W | {result["generation_time_sec"]:.1f}s')
    
    matching = [r for r in video_results if r.get('frames')==frames and r.get('steps')==steps and r.get('resolution')==f'{res}x{res}']
    if matching:
        print_summary(matching)

save_results(video_results, f'exp3_video_{GPU_NAME.replace(" ", "_")}.json')
del pipe, adapter; gc.collect(); torch.cuda.empty_cache()
print('\n✅ Video experiment done!')

In [ ]:
# ============================================================
# Cell 8: EXPERIMENT 4 — MUSIC GENERATION
# ============================================================
from transformers import AutoProcessor, MusicgenForConditionalGeneration

print('\n' + '='*60)
print('🎵 EXPERIMENT 4: MUSIC GENERATION')
print('='*60)

prompt = 'upbeat electronic dance music with a catchy melody'
token_configs = [128, 256, 512]
music_results = []
pm = NVIDIAPowerMonitor()

# --- MusicGen-small ---
print('\n  🎵 Loading MusicGen-small...')
processor = AutoProcessor.from_pretrained('facebook/musicgen-small')
model = MusicgenForConditionalGeneration.from_pretrained('facebook/musicgen-small').to(DEVICE)
params = sum(p.numel() for p in model.parameters())
print(f'     Params: {params/1e6:.0f}M')

# Warm up
inp = processor(text=[prompt], padding=True, return_tensors='pt').to(DEVICE)
with torch.no_grad(): model.generate(**inp, max_new_tokens=32)
torch.cuda.synchronize()

for max_tok in token_configs:
    print(f'\n  📏 MusicGen-small, {max_tok} tokens × {REPEATS} runs')
    for run in range(REPEATS):
        def task(mt=max_tok):
            inp = processor(text=[prompt], padding=True, return_tensors='pt').to(DEVICE)
            with torch.no_grad():
                audio = model.generate(**inp, max_new_tokens=mt)
            torch.cuda.synchronize()
            audio_len = audio.shape[-1] / model.config.audio_encoder.sampling_rate
            return {'audio_sec': round(audio_len, 1), 'max_tokens': mt}
        
        result = run_and_measure(task, pm, cooldown=2)
        result['modality'] = 'music'
        result['model'] = 'MusicGen-small'
        result['params_M'] = round(params/1e6, 0)
        result['max_tokens'] = max_tok
        result['run'] = run + 1
        result['hardware'] = GPU_NAME
        result['backend'] = 'cuda'
        music_results.append(result)
        
        if (run+1) % 10 == 0 or run == 0:
            print(f'    Run {run+1}/{REPEATS}: {result["total_energy_j"]:.0f}J | {result["avg_power_w"]:.1f}W | {result["generation_time_sec"]:.1f}s')
    
    print_summary([r for r in music_results if r['model']=='MusicGen-small' and r['max_tokens']==max_tok])

del model, processor; gc.collect(); torch.cuda.empty_cache()

# --- MusicGen-medium ---
print('\n  🎵 Loading MusicGen-medium...')
processor = AutoProcessor.from_pretrained('facebook/musicgen-medium')
model = MusicgenForConditionalGeneration.from_pretrained('facebook/musicgen-medium').to(DEVICE)
params = sum(p.numel() for p in model.parameters())
print(f'     Params: {params/1e6:.0f}M')

inp = processor(text=[prompt], padding=True, return_tensors='pt').to(DEVICE)
with torch.no_grad(): model.generate(**inp, max_new_tokens=32)
torch.cuda.synchronize()

for max_tok in [128, 256]:
    print(f'\n  📏 MusicGen-medium, {max_tok} tokens × {REPEATS} runs')
    for run in range(REPEATS):
        def task(mt=max_tok):
            inp = processor(text=[prompt], padding=True, return_tensors='pt').to(DEVICE)
            with torch.no_grad():
                audio = model.generate(**inp, max_new_tokens=mt)
            torch.cuda.synchronize()
            audio_len = audio.shape[-1] / model.config.audio_encoder.sampling_rate
            return {'audio_sec': round(audio_len, 1), 'max_tokens': mt}
        
        result = run_and_measure(task, pm, cooldown=2)
        result['modality'] = 'music'
        result['model'] = 'MusicGen-medium'
        result['params_M'] = round(params/1e6, 0)
        result['max_tokens'] = max_tok
        result['run'] = run + 1
        result['hardware'] = GPU_NAME
        result['backend'] = 'cuda'
        music_results.append(result)
        
        if (run+1) % 10 == 0 or run == 0:
            print(f'    Run {run+1}/{REPEATS}: {result["total_energy_j"]:.0f}J | {result["avg_power_w"]:.1f}W | {result["generation_time_sec"]:.1f}s')
    
    print_summary([r for r in music_results if r['model']=='MusicGen-medium' and r['max_tokens']==max_tok])

save_results(music_results, f'exp4_music_{GPU_NAME.replace(" ", "_")}.json')
del model, processor; gc.collect(); torch.cuda.empty_cache()
print('\n✅ Music experiment done!')

In [ ]:
# ============================================================
# Cell 9: Summary + Final Save
# ============================================================
import glob

print('\n' + '='*60)
print('✅ ALL EXPERIMENTS COMPLETE!')
print('='*60)
print(f'\n📁 Results saved to: {RESULTS_DIR}')

for f in sorted(glob.glob(os.path.join(RESULTS_DIR, '*.json'))):
    size = os.path.getsize(f)
    with open(f) as fh:
        data = json.load(fh)
    print(f'  📄 {os.path.basename(f)} — {len(data)} records, {size/1024:.0f}KB')

# Combine all into one summary
summary = {
    'hardware': GPU_NAME,
    'backend': 'cuda',
    'repeats': REPEATS,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'pytorch': torch.__version__,
    'cuda': torch.version.cuda,
}

# Quick per-modality summary
for mod_name, results_list in [('text', text_results), ('image', image_results), 
                                ('video', video_results), ('music', music_results)]:
    energies = [r['total_energy_j'] for r in results_list if r.get('total_energy_j', 0) > 0]
    summary[mod_name] = {
        'n_records': len(results_list),
        'energy_range_j': [round(min(energies), 1), round(max(energies), 1)] if energies else [],
        'energy_mean_j': round(np.mean(energies), 1) if energies else 0,
    }
    print(f'\n  {mod_name.upper()}: {len(results_list)} records, '
          f'energy range {min(energies):.0f}–{max(energies):.0f}J')

with open(os.path.join(RESULTS_DIR, f'summary_{GPU_NAME.replace(" ", "_")}.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\n🎉 Done! Download results from Google Drive: {RESULTS_DIR}')